In [53]:
# Cell 1: Install required libraries

!pip install -q transformers==4.57.1 datasets accelerate librosa soundfile scikit-learn openai-whisper

print("All required libraries installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
All required libraries installed successfully!


In [54]:
# Cell 2: Import libraries and check GPU

import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoFeatureExtractor,
    AutoModelForTokenClassification,
    WavLMModel
)

import whisper

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [55]:
# Cell 3: Load the DisfluencySpeech dataset

print("Loading dataset...")

dataset = load_dataset("amaai-lab/DisfluencySpeech")

print("\nDataset loaded successfully!")
print(dataset)

Loading dataset...

Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcript_annotated', 'transcript_a', 'transcript_b', 'transcript_c'],
        num_rows: 4500
    })
    validation: Dataset({
        features: ['audio', 'transcript_annotated', 'transcript_a', 'transcript_b', 'transcript_c'],
        num_rows: 250
    })
    test: Dataset({
        features: ['audio', 'transcript_annotated', 'transcript_a', 'transcript_b', 'transcript_c'],
        num_rows: 250
    })
})


In [56]:
# Cell 4: Check dataset structure

print("Dataset splits:")
print(dataset)

print("\nTraining columns:")
print(dataset["train"].column_names)

print("\nFirst training example:")
print(dataset["train"][0])

Dataset splits:
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcript_annotated', 'transcript_a', 'transcript_b', 'transcript_c'],
        num_rows: 4500
    })
    validation: Dataset({
        features: ['audio', 'transcript_annotated', 'transcript_a', 'transcript_b', 'transcript_c'],
        num_rows: 250
    })
    test: Dataset({
        features: ['audio', 'transcript_annotated', 'transcript_a', 'transcript_b', 'transcript_c'],
        num_rows: 250
    })
})

Training columns:
['audio', 'transcript_annotated', 'transcript_a', 'transcript_b', 'transcript_c']

First training example:
{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7ab6cb1b6dd0>, 'transcript_annotated': "Yeah, I do <laughter>. Yes, {F uh, } I don't work, though, {C but } I used to work [ and, + ] when I had two children.", 'transcript_a': "Yeah, I do. Yes, uh, I don't work, though, but I used to work and, when I had two children.", 'transcript_b': "Yeah, I do. Yes, I don't wor

In [57]:
# Cell 5: Create word-level disfluency labels

from difflib import SequenceMatcher
import re

def normalize_word(word):
    return re.sub(r"[^\w'-]", "", word.lower())


def create_word_labels(original_text, clean_text):

    # Split transcripts into words
    original_words = re.findall(r"\b[\w'-]+\b", original_text)
    clean_words = re.findall(r"\b[\w'-]+\b", clean_text)

    # Normalize words for comparison
    original_norm = [normalize_word(w) for w in original_words]
    clean_norm = [normalize_word(w) for w in clean_words]

    # Compare original and clean transcripts
    matcher = SequenceMatcher(
        None,
        original_norm,
        clean_norm
    )

    # Initially mark every word as DISFLUENT
    labels = ["DISFLUENT"] * len(original_words)

    # Words that remain in the clean transcript are FLUENT
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        if tag == "equal":

            for i in range(i1, i2):
                labels[i] = "FLUENT"

    return original_words, labels


# Test on the first training example

original_text = dataset["train"][0]["transcript_a"]
clean_text = dataset["train"][0]["transcript_c"]

words, labels = create_word_labels(
    original_text,
    clean_text
)

print("Original transcript:")
print(original_text)

print("\nClean transcript:")
print(clean_text)

print("\nWord-level labels:")

for word, label in zip(words, labels):
    print(f"{word:15} -> {label}")

Original transcript:
Yeah, I do. Yes, uh, I don't work, though, but I used to work and, when I had two children.

Clean transcript:
Yeah, I do. Yes, I don't work, though, but I used to work when I had two children.

Word-level labels:
Yeah            -> FLUENT
I               -> FLUENT
do              -> FLUENT
Yes             -> FLUENT
uh              -> DISFLUENT
I               -> FLUENT
don't           -> FLUENT
work            -> FLUENT
though          -> FLUENT
but             -> FLUENT
I               -> FLUENT
used            -> FLUENT
to              -> FLUENT
work            -> FLUENT
and             -> DISFLUENT
when            -> FLUENT
I               -> FLUENT
had             -> FLUENT
two             -> FLUENT
children        -> FLUENT


In [58]:
# Cell 6: Create word-level labels for all dataset samples

def prepare_split(split):

    all_words = []
    all_labels = []

    for example in dataset[split]:

        original_text = example["transcript_a"]
        clean_text = example["transcript_c"]

        words, labels = create_word_labels(
            original_text,
            clean_text
        )

        all_words.append(words)
        all_labels.append(labels)

    return all_words, all_labels


# Prepare training data
train_words, train_labels = prepare_split("train")

# Prepare validation data
validation_words, validation_labels = prepare_split("validation")

# Prepare test data
test_words, test_labels = prepare_split("test")


print("Label preparation completed!")

print("\nNumber of training samples:", len(train_words))
print("Number of validation samples:", len(validation_words))
print("Number of test samples:", len(test_words))

Label preparation completed!

Number of training samples: 4500
Number of validation samples: 250
Number of test samples: 250


In [59]:
# Cell 7: Check FLUENT and DISFLUENT label counts

from collections import Counter

train_count = Counter(
    label
    for labels in train_labels
    for label in labels
)

validation_count = Counter(
    label
    for labels in validation_labels
    for label in labels
)

test_count = Counter(
    label
    for labels in test_labels
    for label in labels
)


print("Training labels:")
print(train_count)

print("\nValidation labels:")
print(validation_count)

print("\nTest labels:")
print(test_count)

Training labels:
Counter({'FLUENT': 89766, 'DISFLUENT': 12414})

Validation labels:
Counter({'FLUENT': 4863, 'DISFLUENT': 753})

Test labels:
Counter({'FLUENT': 4819, 'DISFLUENT': 660})


In [60]:
# Cell 8: Load XLM-RoBERTa tokenizer

from transformers import AutoTokenizer

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("XLM-RoBERTa tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)


XLM-RoBERTa tokenizer loaded successfully!
Vocabulary size: 250002


In [61]:
# Cell 9: Prepare text and labels for XLM-RoBERTa

label2id = {
    "FLUENT": 0,
    "DISFLUENT": 1
}

id2label = {
    0: "FLUENT",
    1: "DISFLUENT"
}

print("Labels:")
print(label2id)

print("\nExample training sentence:")
print(train_words[0])

print("\nExample labels:")
print(train_labels[0])

Labels:
{'FLUENT': 0, 'DISFLUENT': 1}

Example training sentence:
['Yeah', 'I', 'do', 'Yes', 'uh', 'I', "don't", 'work', 'though', 'but', 'I', 'used', 'to', 'work', 'and', 'when', 'I', 'had', 'two', 'children']

Example labels:
['FLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'DISFLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'DISFLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'FLUENT', 'FLUENT']


In [62]:
# Cell 10: Tokenize words and align labels

def tokenize_and_align_labels(words, labels):

    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=256
    )

    word_ids = tokenized.word_ids()

    aligned_labels = []

    previous_word_id = None

    for word_id in word_ids:

        if word_id is None:
            aligned_labels.append(-100)

        elif word_id != previous_word_id:
            aligned_labels.append(label2id[labels[word_id]])

        else:
            # Ignore additional subword tokens
            aligned_labels.append(-100)

        previous_word_id = word_id

    tokenized["labels"] = aligned_labels

    return tokenized


# Test on the first training example

test_tokenized = tokenize_and_align_labels(
    train_words[0],
    train_labels[0]
)

print("Input IDs:")
print(test_tokenized["input_ids"])

print("\nLabels:")
print(test_tokenized["labels"])

print("\nNumber of tokens:", len(test_tokenized["input_ids"]))
print("Number of labels:", len(test_tokenized["labels"]))

Input IDs:
[0, 119057, 87, 54, 32635, 38074, 87, 2301, 25, 18, 4488, 21208, 1284, 87, 11814, 47, 4488, 136, 3229, 87, 1902, 6626, 20020, 2]

Labels:
[-100, 0, 0, 0, 0, 1, 0, 0, -100, -100, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, -100]

Number of tokens: 24
Number of labels: 24


In [63]:
# Cell 11: Tokenize the complete training, validation, and test data

from datasets import Dataset

def create_tokenized_dataset(words_list, labels_list):

    data = []

    for words, labels in zip(words_list, labels_list):

        tokenized = tokenize_and_align_labels(
            words,
            labels
        )

        data.append(tokenized)

    return Dataset.from_list(data)


print("Tokenizing training data...")
tokenized_train = create_tokenized_dataset(
    train_words,
    train_labels
)

print("Tokenizing validation data...")
tokenized_validation = create_tokenized_dataset(
    validation_words,
    validation_labels
)

print("Tokenizing test data...")
tokenized_test = create_tokenized_dataset(
    test_words,
    test_labels
)

print("\nTokenization completed!")

print("Training samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_validation))
print("Test samples:", len(tokenized_test))

Tokenizing training data...
Tokenizing validation data...
Tokenizing test data...

Tokenization completed!
Training samples: 4500
Validation samples: 250
Test samples: 250


In [64]:
# Cell 12: Load XLM-RoBERTa for disfluency classification

from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("XLM-RoBERTa model loaded successfully!")
print("Device:", device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLM-RoBERTa model loaded successfully!
Device: cuda


In [65]:
# Cell 13: Set up XLM-RoBERTa training

from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):

        for p, l in zip(prediction, label):

            if l != -100:
                true_predictions.append(p)
                true_labels.append(l)

    accuracy = accuracy_score(
        true_labels,
        true_predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


training_args = TrainingArguments(
    output_dir="./xlmr_disfluency_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available()
)

print("Training setup completed!")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)

Training setup completed!
Epochs: 3
Batch size: 8
Learning rate: 2e-05


In [68]:
# Cell 14: Fix batching with dynamic padding

from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer recreated successfully with dynamic padding!")

Trainer recreated successfully with dynamic padding!


In [69]:
# Cell 15: Train XLM-RoBERTa

print("Starting training...")

training_result = trainer.train()

print("\nTraining completed!")

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.105837,0.099454,0.960114,0.845752,0.859230,0.852437
2,0.082678,0.105256,0.963141,0.938907,0.775564,0.849455
3,0.064638,0.096306,0.965812,0.893408,0.845950,0.869031


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed!


In [70]:
# Cell 16: Evaluate XLM-RoBERTa on the test set

print("Evaluating on test data...")

test_results = trainer.evaluate(
    eval_dataset=tokenized_test
)

print("\nTest Results:")

for key, value in test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Evaluating on test data...


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.064638,0.088367,3,0.965687,0.867601,0.843939,0.855607



Test Results:
eval_loss: 0.0884
eval_accuracy: 0.9657
eval_precision: 0.8676
eval_recall: 0.8439
eval_f1: 0.8556


In [71]:
# Cell 17: Load Whisper ASR

import whisper

print("Loading Whisper ASR model...")

whisper_model = whisper.load_model("base")

print("Whisper ASR loaded successfully!")

Loading Whisper ASR model...
Whisper ASR loaded successfully!


In [72]:
# Cell 18: Transcribe one dataset audio sample using Whisper

import soundfile as sf

# Get the first audio sample
audio_data = dataset["test"][0]["audio"]

# Get audio array and sampling rate
audio_array = audio_data.get_all_samples().data.numpy().squeeze()
sample_rate = audio_data.metadata.sample_rate

print("Audio sample rate:", sample_rate)
print("Audio length:", len(audio_array), "samples")

# Whisper expects 16 kHz audio
if sample_rate != 16000:
    audio_array = librosa.resample(
        audio_array,
        orig_sr=sample_rate,
        target_sr=16000
    )
    sample_rate = 16000

print("Sample rate used by Whisper:", sample_rate)

# Transcribe
result = whisper_model.transcribe(
    audio_array,
    language="en"
)

whisper_transcript = result["text"].strip()

print("\nWhisper transcript:")
print(whisper_transcript)

print("\nDataset original transcript:")
print(dataset["test"][0]["transcript_a"])

Audio sample rate: 22050
Audio length: 172940 samples
Sample rate used by Whisper: 16000

Whisper transcript:
Yeah, well, rabbits are darling. That would be fun if you could get them trained. Otherwise, they're pretty smelly.

Dataset original transcript:
Yeah. Well, rabbits are darling. That would be fun if you could get them trained. Otherwise they're pretty smelly.


In [73]:
# Cell 19: Compare Whisper transcript with dataset transcript

print("Whisper ASR output:")
print(whisper_transcript)

print("\n" + "=" * 60)

print("Dataset transcript:")
print(dataset["test"][0]["transcript_a"])

print("\n" + "=" * 60)

print("Dataset clean transcript:")
print(dataset["test"][0]["transcript_c"])

Whisper ASR output:
Yeah, well, rabbits are darling. That would be fun if you could get them trained. Otherwise, they're pretty smelly.

Dataset transcript:
Yeah. Well, rabbits are darling. That would be fun if you could get them trained. Otherwise they're pretty smelly.

Dataset clean transcript:
Yeah. rabbits are darling. That would be fun if you could get them trained. Otherwise they're pretty smelly.


In [74]:
# Cell 20: Use trained XLM-RoBERTa on Whisper transcript

def predict_disfluencies(text):

    # Split transcript into words
    words = re.findall(r"\b[\w'-]+\b", text)

    # Tokenize words
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    # Move inputs to GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Predict
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(
        outputs.logits,
        dim=2
    )[0].cpu().numpy()

    # Map predictions back to words
    word_ids = inputs["input_ids"].new_zeros(
        inputs["input_ids"].shape,
        dtype=torch.long
    )

    # Get word IDs again from tokenizer
    word_id_list = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=256
    ).word_ids()

    word_predictions = {}

    for token_index, word_id in enumerate(word_id_list):

        if word_id is not None and word_id not in word_predictions:
            word_predictions[word_id] = predictions[token_index]

    # Create result
    results = []

    for i, word in enumerate(words):
        label_id = word_predictions.get(i, 0)
        label = id2label[label_id]

        results.append((word, label))

    return results


# Run XLM-R on Whisper's transcript

prediction_results = predict_disfluencies(
    whisper_transcript
)

print("XLM-R predictions:\n")

for word, label in prediction_results:
    print(f"{word:15} -> {label}")

XLM-R predictions:

Yeah            -> FLUENT
well            -> DISFLUENT
rabbits         -> FLUENT
are             -> FLUENT
darling         -> FLUENT
That            -> FLUENT
would           -> FLUENT
be              -> FLUENT
fun             -> FLUENT
if              -> FLUENT
you             -> FLUENT
could           -> FLUENT
get             -> FLUENT
them            -> FLUENT
trained         -> FLUENT
Otherwise       -> FLUENT
they're         -> FLUENT
pretty          -> FLUENT
smelly          -> FLUENT


In [75]:
# Cell 21: Remove predicted disfluencies

def remove_disfluencies(prediction_results):

    clean_words = []

    for word, label in prediction_results:

        if label == "FLUENT":
            clean_words.append(word)

    return " ".join(clean_words)


# Create cleaned transcript
clean_whisper_transcript = remove_disfluencies(
    prediction_results
)

print("Whisper transcript:")
print(whisper_transcript)

print("\nCleaned transcript:")
print(clean_whisper_transcript)

Whisper transcript:
Yeah, well, rabbits are darling. That would be fun if you could get them trained. Otherwise, they're pretty smelly.

Cleaned transcript:
Yeah rabbits are darling That would be fun if you could get them trained Otherwise they're pretty smelly


In [76]:
# Cell 22: Test the complete ASR -> XLM-R pipeline

num_samples = 10

print("Testing complete pipeline on", num_samples, "samples...\n")

for i in range(num_samples):

    print("=" * 70)
    print("Sample", i + 1)

    # Get audio
    audio_data = dataset["test"][i]["audio"]

    audio_array = audio_data.get_all_samples().data.numpy().squeeze()
    sample_rate = audio_data.metadata.sample_rate

    # Resample to 16 kHz
    if sample_rate != 16000:
        audio_array = librosa.resample(
            audio_array,
            orig_sr=sample_rate,
            target_sr=16000
        )

    # Whisper ASR
    result = whisper_model.transcribe(
        audio_array,
        language="en"
    )

    asr_text = result["text"].strip()

    # XLM-R disfluency detection
    predictions = predict_disfluencies(asr_text)

    # Remove disfluencies
    cleaned_text = remove_disfluencies(predictions)

    # Dataset clean transcript
    reference_text = dataset["test"][i]["transcript_c"]

    print("\nWhisper:")
    print(asr_text)

    print("\nXLM-R Cleaned:")
    print(cleaned_text)

    print("\nReference Clean:")
    print(reference_text)

Testing complete pipeline on 10 samples...

Sample 1

Whisper:
Yeah, well, rabbits are darling. That would be fun if you could get them trained. Otherwise, they're pretty smelly.

XLM-R Cleaned:
Yeah rabbits are darling That would be fun if you could get them trained Otherwise they're pretty smelly

Reference Clean:
Yeah. rabbits are darling. That would be fun if you could get them trained. Otherwise they're pretty smelly.
Sample 2

Whisper:
Well, Mac knew, knew, and Lara. I guess it's leaving. One of them is leaving.

XLM-R Cleaned:
Mac knew knew and Lara I guess it's leaving One of them is leaving

Reference Clean:
McNeil, and Lehrer, I guess he's leaving, one of them is leaving.
Sample 3

Whisper:
Well, I don't want to, I don't want this to be a unit letharo conversation, but yes, we do.

XLM-R Cleaned:
I want to I don't want this to be a unit letharo conversation but yes we do

Reference Clean:
I don't want this to be a unilateral conversation but, yes, we do.
Sample 4

Whisper:
We

In [77]:
# Cell 23: Evaluate Whisper -> XLM-R pipeline on the full test set

from difflib import SequenceMatcher
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def text_similarity(reference, prediction):

    reference = normalize_text(reference)
    prediction = normalize_text(prediction)

    return SequenceMatcher(
        None,
        reference,
        prediction
    ).ratio()


similarities = []

print("Evaluating full test set...")
print("Total samples:", len(dataset["test"]))

for i in range(len(dataset["test"])):

    # Get audio
    audio_data = dataset["test"][i]["audio"]

    audio_array = audio_data.get_all_samples().data.numpy().squeeze()
    sample_rate = audio_data.metadata.sample_rate

    # Resample to 16 kHz
    if sample_rate != 16000:
        audio_array = librosa.resample(
            audio_array,
            orig_sr=sample_rate,
            target_sr=16000
        )

    # Whisper ASR
    result = whisper_model.transcribe(
        audio_array,
        language="en"
    )

    asr_text = result["text"].strip()

    # XLM-R
    predictions = predict_disfluencies(asr_text)

    # Remove disfluencies
    cleaned_text = remove_disfluencies(predictions)

    # Reference clean transcript
    reference_text = dataset["test"][i]["transcript_c"]

    # Similarity
    score = text_similarity(
        reference_text,
        cleaned_text
    )

    similarities.append(score)

    if (i + 1) % 25 == 0:
        print(f"Completed {i + 1}/{len(dataset['test'])}")


average_similarity = np.mean(similarities)

print("\n" + "=" * 60)
print("FULL TEST RESULTS")
print("=" * 60)

print(
    f"Average text similarity: "
    f"{average_similarity * 100:.2f}%"
)

Evaluating full test set...
Total samples: 250
Completed 25/250
Completed 50/250
Completed 75/250
Completed 100/250
Completed 125/250
Completed 150/250
Completed 175/250
Completed 200/250
Completed 225/250
Completed 250/250

FULL TEST RESULTS
Average text similarity: 96.21%


In [78]:
# Cell 24: Load WavLM

from transformers import WavLMModel, AutoFeatureExtractor

print("Loading WavLM...")

wavlm_name = "microsoft/wavlm-base"

wavlm_processor = AutoFeatureExtractor.from_pretrained(
    wavlm_name
)

wavlm_model = WavLMModel.from_pretrained(
    wavlm_name
)

wavlm_model.to(device)
wavlm_model.eval()

print("WavLM loaded successfully!")
print("Device:", device)

Loading WavLM...


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

WavLM loaded successfully!
Device: cuda


In [79]:
# Cell 25: Extract WavLM features from one audio sample

# Get the first test audio
audio_data = dataset["test"][0]["audio"]

audio_array = audio_data.get_all_samples().data.numpy().squeeze()
sample_rate = audio_data.metadata.sample_rate

# Resample to 16 kHz
if sample_rate != 16000:
    audio_array = librosa.resample(
        audio_array,
        orig_sr=sample_rate,
        target_sr=16000
    )

# Prepare audio for WavLM
inputs = wavlm_processor(
    audio_array,
    sampling_rate=16000,
    return_tensors="pt"
)

# Move input to GPU
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Extract WavLM features
with torch.no_grad():
    wavlm_output = wavlm_model(**inputs)

audio_features = wavlm_output.last_hidden_state

print("WavLM feature shape:")
print(audio_features.shape)

WavLM feature shape:
torch.Size([1, 391, 768])


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


In [80]:
# Cell 26: Check WavLM features with the corresponding transcript

print("Audio feature shape:")
print(audio_features.shape)

print("\nNumber of WavLM frames:")
print(audio_features.shape[1])

print("\nFeature size:")
print(audio_features.shape[2])

print("\nWhisper transcript:")
print(whisper_transcript)

print("\nNumber of words in Whisper transcript:")

whisper_words = re.findall(
    r"\b[\w'-]+\b",
    whisper_transcript
)

print(len(whisper_words))
print(whisper_words)

Audio feature shape:
torch.Size([1, 391, 768])

Number of WavLM frames:
391

Feature size:
768

Whisper transcript:
Yeah, well, rabbits are darling. That would be fun if you could get them trained. Otherwise, they're pretty smelly.

Number of words in Whisper transcript:
19
['Yeah', 'well', 'rabbits', 'are', 'darling', 'That', 'would', 'be', 'fun', 'if', 'you', 'could', 'get', 'them', 'trained', 'Otherwise', "they're", 'pretty', 'smelly']


In [81]:
# Cell 27: Create a single audio representation using WavLM

# Average all WavLM frames
audio_embedding = audio_features.mean(
    dim=1
)

print("Original WavLM features:")
print(audio_features.shape)

print("\nAudio embedding shape:")
print(audio_embedding.shape)

Original WavLM features:
torch.Size([1, 391, 768])

Audio embedding shape:
torch.Size([1, 768])


In [82]:
# Cell 28: Check the WavLM audio embedding

print("Audio embedding shape:")
print(audio_embedding.shape)

print("\nFirst 10 audio features:")
print(audio_embedding[0, :10])

Audio embedding shape:
torch.Size([1, 768])

First 10 audio features:
tensor([-0.1006, -0.0125, -0.1014, -0.0430,  0.0322,  0.0568, -0.0895, -0.1244,
         0.0451, -0.0375], device='cuda:0')


In [83]:
# Cell 29: Extract XLM-R text representation

# Tokenize Whisper transcript
text_inputs = tokenizer(
    whisper_transcript,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

# Move inputs to GPU
text_inputs = {
    key: value.to(device)
    for key, value in text_inputs.items()
}

# Use the trained XLM-R model
with torch.no_grad():
    text_outputs = model.base_model(**text_inputs)

# Average token representations
text_embedding = text_outputs.last_hidden_state.mean(
    dim=1
)

print("XLM-R text embedding shape:")
print(text_embedding.shape)

print("\nFirst 10 text features:")
print(text_embedding[0, :10])

XLM-R text embedding shape:
torch.Size([1, 768])

First 10 text features:
tensor([-0.0608,  0.1834, -0.8888,  0.2708, -0.0179,  0.1195,  0.1011, -0.7501,
        -0.2997, -2.1607], device='cuda:0')


In [84]:
# Cell 30: Combine WavLM audio features and XLM-R text features

# Concatenate audio and text embeddings
combined_embedding = torch.cat(
    [audio_embedding, text_embedding],
    dim=1
)

print("WavLM embedding shape:")
print(audio_embedding.shape)

print("\nXLM-R embedding shape:")
print(text_embedding.shape)

print("\nCombined embedding shape:")
print(combined_embedding.shape)

WavLM embedding shape:
torch.Size([1, 768])

XLM-R embedding shape:
torch.Size([1, 768])

Combined embedding shape:
torch.Size([1, 1536])


In [85]:
# Cell 31: Create WavLM + XLM-R fusion classifier

import torch.nn as nn

class FusionClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(1536, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.classifier(x)


fusion_model = FusionClassifier().to(device)

print("Fusion classifier created successfully!")
print(fusion_model)

Fusion classifier created successfully!
FusionClassifier(
  (classifier): Sequential(
    (0): Linear(in_features=1536, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=2, bias=True)
  )
)


In [86]:
# Cell 32: Check word-level labels for the first sample

first_words = train_words[0]
first_labels = train_labels[0]

print("Words and labels:\n")

for word, label in zip(first_words, first_labels):
    print(f"{word:15} -> {label}")

Words and labels:

Yeah            -> FLUENT
I               -> FLUENT
do              -> FLUENT
Yes             -> FLUENT
uh              -> DISFLUENT
I               -> FLUENT
don't           -> FLUENT
work            -> FLUENT
though          -> FLUENT
but             -> FLUENT
I               -> FLUENT
used            -> FLUENT
to              -> FLUENT
work            -> FLUENT
and             -> DISFLUENT
when            -> FLUENT
I               -> FLUENT
had             -> FLUENT
two             -> FLUENT
children        -> FLUENT


In [87]:
# Cell 33: Create utterance-level labels

def create_utterance_labels(word_labels):
    labels = []

    for labels_for_sample in word_labels:
        if "DISFLUENT" in labels_for_sample:
            labels.append(1)   # DISFLUENT
        else:
            labels.append(0)   # FLUENT

    return labels


train_utterance_labels = create_utterance_labels(train_labels)
val_utterance_labels = create_utterance_labels(val_labels)
test_utterance_labels = create_utterance_labels(test_labels)

print("Train utterance labels:", len(train_utterance_labels))
print("Validation utterance labels:", len(val_utterance_labels))
print("Test utterance labels:", len(test_utterance_labels))

print("\nFirst 10 train labels:")
print(train_utterance_labels[:10])

Train utterance labels: 4500
Validation utterance labels: 250
Test utterance labels: 250

First 10 train labels:
[1, 0, 1, 1, 0, 1, 1, 1, 1, 1]


In [88]:
# Cell 34: Function to extract WavLM audio embedding

def get_wavlm_embedding(audio_array, sampling_rate=16000):

    # Convert audio to float32
    audio_array = np.asarray(audio_array, dtype=np.float32)

    # Process audio
    inputs = wavlm_processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Extract WavLM features
    with torch.no_grad():
        outputs = wavlm_model(**inputs)

    # Average all audio frames
    embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding


print("WavLM embedding function created successfully!")

WavLM embedding function created successfully!


In [89]:
# Cell 35: Function to extract XLM-R text embedding

def get_xlmr_embedding(text):

    # Tokenize text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Extract XLM-R features
    with torch.no_grad():
        outputs = model.base_model(**inputs)

    # Average token representations
    embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding


print("XLM-R embedding function created successfully!")

XLM-R embedding function created successfully!


In [92]:
# Cell 36: Check available dataset variables

print("Checking dataset variables...")

for name in ["dataset", "ds", "train_data", "train_dataset"]:
    if name in globals():
        print(name, "is available")

Checking dataset variables...
dataset is available


In [93]:
# Cell 37: Test WavLM + XLM-R embeddings

# Get first training sample
sample = dataset["train"][0]

sample_audio = sample["audio"]["array"]
sample_text = sample["transcript_a"]

# Get WavLM embedding
sample_audio_embedding = get_wavlm_embedding(
    sample_audio,
    sampling_rate=16000
)

# Get XLM-R embedding
sample_text_embedding = get_xlmr_embedding(sample_text)

# Combine both embeddings
sample_combined_embedding = torch.cat(
    [sample_audio_embedding, sample_text_embedding],
    dim=1
)

print("WavLM embedding shape:", sample_audio_embedding.shape)
print("XLM-R embedding shape:", sample_text_embedding.shape)
print("Combined embedding shape:", sample_combined_embedding.shape)

WavLM embedding shape: torch.Size([1, 768])
XLM-R embedding shape: torch.Size([1, 768])
Combined embedding shape: torch.Size([1, 1536])


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


In [94]:
# Cell 38: Extract WavLM + XLM-R features for training data

train_fusion_features = []
train_fusion_labels = []

for i in range(len(dataset["train"])):

    sample = dataset["train"][i]

    # Get audio and text
    audio_array = sample["audio"]["array"]
    text = sample["transcript_a"]

    # WavLM embedding
    audio_embedding = get_wavlm_embedding(
        audio_array,
        sampling_rate=16000
    )

    # XLM-R embedding
    text_embedding = get_xlmr_embedding(text)

    # Combine audio + text
    combined = torch.cat(
        [audio_embedding, text_embedding],
        dim=1
    )

    # Store feature
    train_fusion_features.append(
        combined.squeeze(0).cpu()
    )

    # Store utterance-level label
    train_fusion_labels.append(
        train_utterance_labels[i]
    )

    # Progress
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/4500 samples")

print("\nTraining feature extraction completed!")
print("Number of features:", len(train_fusion_features))
print("Feature size:", train_fusion_features[0].shape)
print("Number of labels:", len(train_fusion_labels))

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 100/4500 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 200/4500 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

...

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 4400/4500 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 4500/4500 samples

Training feature extraction completed!
Number of features: 4500
Feature size: torch.Size([1536])
Number of labels: 4500


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


In [95]:
# Cell 39: Convert training features and labels into tensors

train_fusion_features = torch.stack(train_fusion_features)

train_fusion_labels = torch.tensor(
    train_fusion_labels,
    dtype=torch.long
)

print("Training features shape:")
print(train_fusion_features.shape)

print("\nTraining labels shape:")
print(train_fusion_labels.shape)

print("\nFirst feature shape:")
print(train_fusion_features[0].shape)

print("\nFirst label:")
print(train_fusion_labels[0].item())

Training features shape:
torch.Size([4500, 1536])

Training labels shape:
torch.Size([4500])

First feature shape:
torch.Size([1536])

First label:
1


In [96]:
# Cell 40: Extract validation features

val_fusion_features = []
val_fusion_labels = []

for i in range(len(dataset["validation"])):

    sample = dataset["validation"][i]

    # Get audio and text
    audio_array = sample["audio"]["array"]
    text = sample["transcript_a"]

    # WavLM embedding
    audio_embedding = get_wavlm_embedding(
        audio_array,
        sampling_rate=16000
    )

    # XLM-R embedding
    text_embedding = get_xlmr_embedding(text)

    # Combine audio + text
    combined = torch.cat(
        [audio_embedding, text_embedding],
        dim=1
    )

    # Store feature
    val_fusion_features.append(
        combined.squeeze(0).cpu()
    )

    # Store label
    val_fusion_labels.append(
        val_utterance_labels[i]
    )

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/250 samples")

print("\nValidation feature extraction completed!")
print("Number of features:", len(val_fusion_features))
print("Feature size:", val_fusion_features[0].shape)
print("Number of labels:", len(val_fusion_labels))

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 50/250 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 100/250 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

...

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 200/250 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 250/250 samples

Validation feature extraction completed!
Number of features: 250
Feature size: torch.Size([1536])
Number of labels: 250


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


In [97]:
# Cell 41: Convert validation features and labels into tensors

val_fusion_features = torch.stack(val_fusion_features)

val_fusion_labels = torch.tensor(
    val_fusion_labels,
    dtype=torch.long
)

print("Validation features shape:")
print(val_fusion_features.shape)

print("\nValidation labels shape:")
print(val_fusion_labels.shape)

print("\nFirst validation feature shape:")
print(val_fusion_features[0].shape)

print("\nFirst validation label:")
print(val_fusion_labels[0].item())

Validation features shape:
torch.Size([250, 1536])

Validation labels shape:
torch.Size([250])

First validation feature shape:
torch.Size([1536])

First validation label:
1


In [98]:
# Cell 42: Create training and validation DataLoaders

from torch.utils.data import TensorDataset, DataLoader

# Create datasets
train_fusion_dataset = TensorDataset(
    train_fusion_features,
    train_fusion_labels
)

val_fusion_dataset = TensorDataset(
    val_fusion_features,
    val_fusion_labels
)

# Create DataLoaders
train_fusion_loader = DataLoader(
    train_fusion_dataset,
    batch_size=32,
    shuffle=True
)

val_fusion_loader = DataLoader(
    val_fusion_dataset,
    batch_size=32,
    shuffle=False
)

print("Training batches:", len(train_fusion_loader))
print("Validation batches:", len(val_fusion_loader))

Training batches: 141
Validation batches: 8


In [99]:
# Cell 43: Setup optimizer and loss function

import torch.nn as nn

# Make sure fusion model is on GPU
fusion_model = fusion_model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    fusion_model.parameters(),
    lr=0.001
)

print("Fusion model training setup completed!")
print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")

Fusion model training setup completed!
Loss function: CrossEntropyLoss
Optimizer: Adam
Learning rate: 0.001


In [100]:
# Cell 44: Train the fusion classifier

num_epochs = 5

for epoch in range(num_epochs):

    # Training mode
    fusion_model.train()

    total_loss = 0

    for features, labels in train_fusion_loader:

        # Move data to GPU
        features = features.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = fusion_model(features)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Update model
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_fusion_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Training Loss: {average_loss:.4f}"
    )

Epoch 1/5 - Training Loss: 0.2369
Epoch 2/5 - Training Loss: 0.1676
Epoch 3/5 - Training Loss: 0.1473
Epoch 4/5 - Training Loss: 0.1431
Epoch 5/5 - Training Loss: 0.1325


In [101]:
# Cell 45: Validate the fusion classifier

fusion_model.eval()

correct = 0
total = 0

with torch.no_grad():

    for features, labels in val_fusion_loader:

        features = features.to(device)
        labels = labels.to(device)

        outputs = fusion_model(features)

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

validation_accuracy = correct / total

print("Validation Accuracy:", validation_accuracy)
print("Validation Accuracy (%):", validation_accuracy * 100)

Validation Accuracy: 0.956
Validation Accuracy (%): 95.6


In [102]:
# Cell 46: Extract WavLM + XLM-R features for test data

test_fusion_features = []
test_fusion_labels = []

for i in range(len(dataset["test"])):

    sample = dataset["test"][i]

    # Get audio and text
    audio_array = sample["audio"]["array"]
    text = sample["transcript_a"]

    # WavLM embedding
    audio_embedding = get_wavlm_embedding(
        audio_array,
        sampling_rate=16000
    )

    # XLM-R embedding
    text_embedding = get_xlmr_embedding(text)

    # Combine audio + text
    combined = torch.cat(
        [audio_embedding, text_embedding],
        dim=1
    )

    # Store feature
    test_fusion_features.append(
        combined.squeeze(0).cpu()
    )

    # Store label
    test_fusion_labels.append(
        test_utterance_labels[i]
    )

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/250 samples")

print("\nTest feature extraction completed!")
print("Number of features:", len(test_fusion_features))
print("Feature size:", test_fusion_features[0].shape)
print("Number of labels:", len(test_fusion_labels))

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 50/250 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 100/250 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 150/250 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 200/250 samples


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

Processed 250/250 samples

Test feature extraction completed!
Number of features: 250
Feature size: torch.Size([1536])
Number of labels: 250


In [103]:
# Cell 47: Convert test features and labels into tensors

test_fusion_features = torch.stack(test_fusion_features)

test_fusion_labels = torch.tensor(
    test_fusion_labels,
    dtype=torch.long
)

print("Test features shape:")
print(test_fusion_features.shape)

print("\nTest labels shape:")
print(test_fusion_labels.shape)

print("\nFirst test feature shape:")
print(test_fusion_features[0].shape)

print("\nFirst test label:")
print(test_fusion_labels[0].item())

Test features shape:
torch.Size([250, 1536])

Test labels shape:
torch.Size([250])

First test feature shape:
torch.Size([1536])

First test label:
1


In [104]:
# Cell 48: Test the fusion classifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

fusion_model.eval()

all_predictions = []
all_true_labels = []

with torch.no_grad():

    for features, labels in DataLoader(
        TensorDataset(test_fusion_features, test_fusion_labels),
        batch_size=32,
        shuffle=False
    ):

        features = features.to(device)
        labels = labels.to(device)

        # Model prediction
        outputs = fusion_model(features)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_true_labels.extend(
            labels.cpu().numpy()
        )

print("Test prediction completed!")
print("Number of predictions:", len(all_predictions))
print("Number of true labels:", len(all_true_labels))

Test prediction completed!
Number of predictions: 250
Number of true labels: 250


In [105]:
# Cell 49: Calculate fusion model performance

accuracy = accuracy_score(
    all_true_labels,
    all_predictions
)

precision = precision_score(
    all_true_labels,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_true_labels,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_true_labels,
    all_predictions,
    zero_division=0
)

print("Fusion Model Test Results")
print("-------------------------")
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1 Score : {f1 * 100:.2f}%")

Fusion Model Test Results
-------------------------
Accuracy : 94.40%
Precision: 96.62%
Recall   : 96.62%
F1 Score : 96.62%


In [106]:
# Cell 50: Compare fusion prediction with word-level XLM-R prediction

# Select one test sample
sample = dataset["test"][0]

# Original transcript
original_text = sample["transcript_a"]

print("Original Transcript:")
print(original_text)

print("\n----------------------------------")

# Word-level XLM-R prediction
word_predictions = predict_disfluencies(original_text)

print("\nWord-level XLM-R Detection:")
for word, label in word_predictions:
    if label == "DISFLUENT":
        print(word, "->", label)

# Fusion model prediction
fusion_feature = test_fusion_features[0].unsqueeze(0).to(device)

fusion_model.eval()

with torch.no_grad():
    fusion_output = fusion_model(fusion_feature)
    fusion_prediction = torch.argmax(
        fusion_output,
        dim=1
    ).item()

print("\n----------------------------------")

if fusion_prediction == 1:
    print("Fusion Model: DISFLUENCY PRESENT")
else:
    print("Fusion Model: NO DISFLUENCY")

Original Transcript:
Yeah. Well, rabbits are darling. That would be fun if you could get them trained. Otherwise they're pretty smelly.

----------------------------------

Word-level XLM-R Detection:
Well -> DISFLUENT

----------------------------------
Fusion Model: DISFLUENCY PRESENT


In [107]:
# Cell 51: Remove detected disfluencies and create clean transcript

cleaned_words = []

for word, label in word_predictions:

    if label == "FLUENT":
        cleaned_words.append(word)

clean_transcript = " ".join(cleaned_words)

print("Original Transcript:")
print(original_text)

print("\nDetected Disfluencies:")
for word, label in word_predictions:
    if label == "DISFLUENT":
        print(word)

print("\nClean Transcript:")
print(clean_transcript)

Original Transcript:
Yeah. Well, rabbits are darling. That would be fun if you could get them trained. Otherwise they're pretty smelly.

Detected Disfluencies:
Well

Clean Transcript:
Yeah rabbits are darling That would be fun if you could get them trained Otherwise they're pretty smelly


In [108]:
# Cell 52: Final end-to-end system test

# Select a test audio sample
sample = dataset["test"][0]

audio_array = sample["audio"]["array"]

print("========== FINAL SPEECH DISFLUENCY SYSTEM ==========\n")

# Step 1: Whisper ASR
audio_input = whisper.pad_or_trim(
    torch.tensor(audio_array, dtype=torch.float32)
)

mel = whisper.log_mel_spectrogram(
    audio_input
).to(whisper_model.device)

with torch.no_grad():
    result = whisper_model.decode(
        mel,
        whisper.DecodingOptions(language="en")
    )

whisper_text = result.text.strip()

print("1. Whisper Transcript:")
print(whisper_text)

# Step 2: WavLM audio features
wavlm_embedding = get_wavlm_embedding(
    audio_array,
    sampling_rate=16000
)

print("\n2. WavLM Feature Shape:")
print(wavlm_embedding.shape)

# Step 3: XLM-R word-level detection
word_predictions = predict_disfluencies(
    whisper_text
)

print("\n3. Detected Disfluencies:")

detected = []

for word, label in word_predictions:
    if label == "DISFLUENT":
        detected.append(word)
        print(word, "-> DISFLUENT")

if len(detected) == 0:
    print("No disfluencies detected")

# Step 4: Remove disfluencies
cleaned_words = [
    word
    for word, label in word_predictions
    if label == "FLUENT"
]

clean_transcript = " ".join(cleaned_words)

print("\n4. Clean Transcript:")
print(clean_transcript)

print("\n========== SYSTEM COMPLETED ==========")

========== FINAL SPEECH DISFLUENCY SYSTEM ==========

1. Whisper Transcript:
Yeah, well, rabbits are darling. That would be fun if you could get them trained. Otherwise, they're pretty smelly.

2. WavLM Feature Shape:
torch.Size([1, 768])

3. Detected Disfluencies:
well -> DISFLUENT

4. Clean Transcript:
Yeah rabbits are darling That would be fun if you could get them trained Otherwise they're pretty smelly

========== SYSTEM COMPLETED ==========


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


In [109]:
# Final evaluation AFTER removing disfluencies

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from difflib import SequenceMatcher
import re

def get_words(text):
    return re.findall(r"\b[\w'-]+\b", text.lower())

all_true_words = []
all_pred_words = []

for i in range(len(dataset["test"])):

    # Original/disfluent transcript
    original_text = dataset["test"][i]["transcript_a"]

    # Actual clean transcript
    clean_text = dataset["test"][i]["transcript_c"]

    # XLM-R detects disfluencies
    predictions = predict_disfluencies(original_text)

    # Remove detected disfluencies
    predicted_clean = remove_disfluencies(predictions)

    true_words = get_words(clean_text)
    pred_words = get_words(predicted_clean)

    # Align predicted words with actual clean words
    matcher = SequenceMatcher(None, true_words, pred_words)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        if tag == "equal":
            # Correctly retained words
            for _ in range(i2 - i1):
                all_true_words.append(1)
                all_pred_words.append(1)

        elif tag == "delete":
            # Missing word in prediction
            for _ in range(i1, i2):
                all_true_words.append(1)
                all_pred_words.append(0)

        elif tag == "insert":
            # Extra word in prediction
            for _ in range(j1, j2):
                all_true_words.append(0)
                all_pred_words.append(1)

        elif tag == "replace":
            n = max(i2 - i1, j2 - j1)
            for k in range(n):
                true_val = 1 if k < (i2 - i1) else 0
                pred_val = 1 if k < (j2 - j1) else 0

                all_true_words.append(true_val)
                all_pred_words.append(pred_val)


# Calculate final metrics
accuracy = accuracy_score(all_true_words, all_pred_words)
precision = precision_score(all_true_words, all_pred_words, zero_division=0)
recall = recall_score(all_true_words, all_pred_words, zero_division=0)
f1 = f1_score(all_true_words, all_pred_words, zero_division=0)

print("==============================================")
print(" FINAL RESULT AFTER REMOVING DISFLUENCIES")
print("==============================================")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print("==============================================")

 FINAL RESULT AFTER REMOVING DISFLUENCIES
Accuracy  : 97.34%
Precision : 98.47%
Recall    : 98.84%
F1-Score  : 98.65%
